In [2]:
#| default_exp storage

In [1]:
#| export
dbhost = '192.168.187.12'
THREADS_NUM = 10
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
engine = create_engine(f'postgresql+psycopg2://postgres:rlp4ZKc6oC0OzgK1FSsJ@{dbhost}:5535', pool_size=THREADS_NUM+2)
session_maker = sessionmaker(bind=engine, autocommit=False)
connection = engine.connect()
from sqlalchemy.exc import IntegrityError
from sqlalchemy.dialects.postgresql import insert, ARRAY, REAL
from sqlalchemy import Table, Column, LargeBinary, DateTime, Integer, String, MetaData, ForeignKey, update, func, delete, select, text, Index

In [4]:
#| export
metadata = MetaData()

In [5]:
#| export
logs = Table('logs', metadata,
     Column('created', DateTime, nullable=False, server_default=func.current_timestamp()),
     Column('ip', String, ),
     Column('origin', String, ),
     Column('agent', String, ),
     Column('fs', String, ),
     Column('ff', String, ),
     Column('session', String, ),
     Column('type', Integer, ),
     Column('hash', String, ),
     Column('log', String, ),
     Column('model', String, ),
)

In [6]:
#| export
ban = Table('ban', metadata,
     Column('created', DateTime, nullable=False, server_default=func.current_timestamp()),
     Column('ip', String, primary_key=True),
     Column('ban_type', Integer, nullable=False, server_default="1"),
     Column('note', String, ),
)
#ban.drop(engine)
#connection.execute(delete(ban))

In [7]:
#connection.execute(delete(ban).where(ban.c.ip=='100.21.134.76'))

In [8]:
#| export
metadata.create_all(engine)
Index("log_created_ix", logs.c.created).create(connection, checkfirst=True)

Index('log_created_ix', Column('created', DateTime(), table=<logs>, nullable=False, server_default=DefaultClause(<sqlalchemy.sql.functions.current_timestamp at 0x7f7b394f9130; current_timestamp>, for_update=False)))

connection.execute("""delete from ban
                   """)

In [13]:
connection.execute("""insert into ban(ip)
                      select ip
                        from logs 
                       where created > clock_timestamp() - interval '48 hours'  
                         and ip != '100.21.134.76'
                         and origin is Null
                       group by ip 
                       having count(*) > 2000
                      except select ip from ban 
                   """)

In [14]:
len(connection.execute(select(ban)).fetchall())

0

In [ ]:
connection.execute("""
select ip,origin,log 
from logs 
where type=0 and ip='185.210.142.231'  
order by created desc
""").fetchall()

In [ ]:
connection.execute("""
select ip,origin,log 
from logs 
where type=0 and created > clock_timestamp() - interval '48 hours'  
order by created desc
""").fetchall()

In [ ]:
select count(*)
from logs 
where type=0 and created > clock_timestamp() - interval '48 hours'  ;

In [14]:
connection.execute("""
select ip from ban 
""").fetchall()


[('176.124.216.85',),
 ('109.252.179.161',),
 ('79.165.74.70',),
 ('162.248.100.50',),
 ('5.145.254.28',),
 ('109.252.176.157',),
 ('185.17.3.106',),
 ('178.20.45.159',),
 ('193.123.61.191',),
 ('92.53.64.103',)]

In [19]:
connection.execute("""
select ip 
                        from logs 
                       where created > clock_timestamp() - interval '48 hours'  
                         and ip != '100.21.134.76'
                       group by ip 
                       having count(*) > 4000
                      except select ip from ban 
""").fetchall()


[('46.150.10.69',),
 ('62.117.111.20',),
 ('2.92.196.91',),
 ('31.47.128.235',),
 ('94.25.106.30',),
 ('178.214.247.118',),
 ('195.216.134.136',),
 ('178.69.80.182',),
 ('213.59.138.6',),
 ('91.123.150.146',),
 ('178.178.88.156',)]

In [ ]:
connection.execute("""
select *
from logs 
where type=0 and ip='195.216.134.136'
order by created desc
""").fetchall()


In [13]:
connection.execute("""
select ip,count(*) c 
from logs 
where type=0 and created > clock_timestamp() - interval '48 hours'  
group by ip
order by c desc
""").fetchall()


[('195.216.134.136', 12584),
 ('94.25.106.30', 2674),
 ('46.150.10.69', 2649),
 ('100.21.134.76', 2407),
 ('2.92.196.91', 2291),
 ('178.178.88.156', 1817),
 ('178.69.80.182', 1484),
 ('178.214.247.118', 1363),
 ('91.123.150.146', 1351),
 ('62.117.111.20', 1207),
 ('31.47.128.235', 1198),
 ('213.59.138.6', 1070),
 ('5.18.216.90', 982),
 ('85.249.23.229', 893),
 ('85.140.8.244', 864),
 ('178.120.6.65', 777),
 ('109.169.211.124', 763),
 ('90.151.137.148', 755),
 ('77.238.132.79', 754),
 ('77.94.196.98', 750),
 ('213.87.163.243', 746),
 ('95.153.180.110', 717),
 ('176.49.132.170', 715),
 ('95.73.50.103', 699),
 ('62.182.75.216', 685),
 ('141.105.135.18', 670),
 ('91.193.179.214', 660),
 ('5.16.120.76', 656),
 ('81.200.10.171', 652),
 ('176.49.94.194', 649),
 ('213.87.127.114', 637),
 ('5.16.105.161', 626),
 ('37.77.110.149', 621),
 ('178.47.80.61', 585),
 ('37.155.67.237', 528),
 ('188.243.182.45', 527),
 ('176.210.31.14', 518),
 ('178.178.88.141', 517),
 ('194.28.153.114', 516),
 ('85.26.